# Training runs: MedViTV2-tiny on DermaMNIST (ELEC-444)
Runs one experiment from our team repo. Results are saved to Google Drive.

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save. Run cells top to bottom.

| Person | Runs (edit Cell 4 for each one) |
|---|---|
| A | `ce` seed 0, 1, 2 |
| B | `weighted inverse` seed 0, `weighted sqrt` seed 0 → then the better one with seed 1 and 2 |
| C | `focal` gamma 1, 2, 3 with seed 0 → then the best gamma with seed 1 and 2 |

Each run takes about **1.5–2 hours**. Keep the tab open. If Colab disconnects, just run all cells again with the **same settings**: training continues where it stopped.

## Cell 1: Check the GPU

In [ ]:
# ===== CELL 1 =====
!nvidia-smi
print('\n>>> CELL 1 DONE. Now run Cell 2.')

## Cell 2: Connect Google Drive
Pick your account, check **Select all**, click Continue.

In [ ]:
# ===== CELL 2 =====
from google.colab import drive
drive.mount('/content/drive')
print('\n>>> CELL 2 DONE. Now run Cell 3.')

## Cell 3: Set up Python 3.11 and get our team code (5–10 minutes)

In [ ]:
# ===== CELL 3 =====
!pip install -q uv
![ -x /content/py311/bin/python ] || uv venv --python 3.11 /content/py311 -q
!uv pip install -q --python /content/py311/bin/python torch==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu124
!uv pip install -q --python /content/py311/bin/python "natten==0.17.5+torch250cu124" -f https://whl.natten.org
!uv pip install -q --python /content/py311/bin/python timm==1.0.15 einops medmnist==3.0.2 numpy pandas scikit-learn tqdm pillow
# Our team repo (always get the latest version of train.py and losses.py)
!rm -rf /content/MedViTV2 && git clone -q https://github.com/Qais-2000/MedViTV2.git /content/MedViTV2
# Model file version that matches NATTEN 0.17 and the reproduced baseline (before the Oct 23, 2026 upstream update)
!cd /content/MedViTV2 && (git checkout -q 4a9e517 -- MedViT.py || (git fetch -q https://github.com/Omid-Nejati/MedViTV2.git && git checkout -q 4a9e517 -- MedViT.py))
!mkdir -p /content/MedViTV2/data
print('\nChecks:')
!/content/py311/bin/python -c "import sys, torch, natten; print('python', sys.version.split()[0], '| torch', torch.__version__, '| natten', natten.__version__, '| GPU ok:', torch.cuda.is_available())"
!cd /content/MedViTV2 && ls train.py losses.py && grep -c is_natten_post_017 MedViT.py
print('\n>>> CELL 3 DONE if you see python 3.11 / torch 2.5.0 / natten 0.17.5 / GPU ok: True, both file names, and a number. Now edit and run Cell 4.')

## Cell 4: Choose your run and train
Change **only** the four lines at the top, then run the cell.
- `LOSS`: `"ce"`, `"weighted"`, or `"focal"`
- `WEIGHTING`: `"inverse"` or `"sqrt"` (only used when LOSS is `"weighted"`)
- `GAMMA`: 1, 2 or 3 (only used when LOSS is `"focal"`)
- `SEED`: 0, 1 or 2

You'll see one line per epoch (50 total). At the end it prints the TEST results and **FINISHED**.

In [ ]:
# ===== CELL 4 =====
LOSS = "ce"
WEIGHTING = "inverse"
GAMMA = 2
SEED = 0

!cd /content/MedViTV2 && /content/py311/bin/python -W ignore train.py --loss {LOSS} --weighting {WEIGHTING} --gamma {GAMMA} --seed {SEED}

## After each run
Results are in Google Drive: `MyDrive/elec444_project/results/<run name>/`
(`log.csv`, `test_metrics.json`, `config.json`, `test_probs.npy`, `best.pth`).

Send Claude the **`log.csv`** and **`test_metrics.json`** of the run, starting your message with `[Person X] [Phase: training] run <name> finished`.
For the next run, only change Cell 4 and run it again (no need to rerun Cells 1–3 if Colab is still connected).